# Aff-Wild2 Stage 3 — bimodal fusion (F0–F5)

Trains and evaluates the six fusion variants from the report against the cached enet visual features and HuBERT-large aligned audio features. Mirrors the interim CREMA-D leaderboard but on the MTL validation split.

Sequence:
1. Train Stage-3 `visual_only` and `audio_only` controls (used by F0).
2. Train F1–F5 (each ~4–6 h on a 3050 Ti).
3. Eval F0 (post-hoc grid blend over the two controls; no training).
4. Eval F1–F5.
5. Build a comparison table that reuses the Sprint A.5 frame-average baseline JSON dumps as the zero-train floor.

Prerequisites:
* `aw2_01_extract_visual.ipynb` populated `cache/features/enet_b0_8_va_mtl/`.
* `aw2_03_extract_audio.ipynb` populated `cache/features/hubert_large/` and bridged annotations into `data/affwild2/annotations/`.
* `aw2_04_align_audio.ipynb` populated `cache/features/hubert_large_aligned/`.

In [1]:
import os, sys, json, subprocess
from pathlib import Path

REPO = Path.cwd().resolve().parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

RESULTS = REPO / 'results'
AW2_S3  = RESULTS / 'aw2_stage3'
AW2_S3.mkdir(parents=True, exist_ok=True)

for d in (REPO / 'cache' / 'features' / 'enet_b0_8_va_mtl',
          REPO / 'cache' / 'features' / 'hubert_large_aligned',
          REPO / 'data'  / 'affwild2' / 'annotations'):
    assert d.exists(), f'missing prerequisite path: {d}'
print('cwd:', Path.cwd())

cwd: C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code


## 1. Train the unimodal controls (`visual_only`, `audio_only`)

These are the F0 inputs and provide the unimodal rows in the comparison table. Both train through `train_fusion.py` so they share the Stage 3 forward pass and bookkeeping.

In [ ]:
def run_train(config_rel):
    cmd = [sys.executable, '-u', '-m', 'src.train_fusion', '--config', config_rel]
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, check=False)
    assert proc.returncode == 0, f'{config_rel} failed (exit={proc.returncode})'

run_train('configs/stage3_visual_only.yaml')
run_train('configs/stage3_audio_only.yaml')

## 2. Train F1–F5

F0 has no learnable parameters of its own and is evaluated post-hoc in step 3.

In [4]:
FUSION_CONFIGS = [
    'configs/stage3_f1_concat.yaml',
    'configs/stage3_f2_blend.yaml',
    'configs/stage3_f3_gate.yaml',
    'configs/stage3_f4_xattn.yaml',
    'configs/stage3_f5_lmf.yaml',
]

for cfg in FUSION_CONFIGS:
    run_train(cfg)

$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u train_fusion.py --config configs/stage3_f1_concat.yaml
$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u train_fusion.py --config configs/stage3_f2_blend.yaml
$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u train_fusion.py --config configs/stage3_f3_gate.yaml
$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u train_fusion.py --config configs/stage3_f4_xattn.yaml
$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u train_fusion.py --config configs/stage3_f5_lmf.yaml


## 3. Evaluate F0 and F1–F5

F0 is the post-hoc grid blend over the trained `visual_only` and `audio_only` checkpoints. F1–F5 are evaluated from their per-variant `best.pt`.

In [5]:
from src.eval_fusion import evaluate_f0_grid, evaluate_variant

f0_metrics = evaluate_f0_grid(
    visual_checkpoint='results/stage3_visual_only/best.pt',
    audio_checkpoint='results/stage3_audio_only/best.pt',
    config_path='configs/stage3_f0_grid.yaml',
)
print('F0:', {k: round(v, 4) if isinstance(v, float) else v for k, v in f0_metrics.items()})

[stage3] wrote results\stage3_f0_grid\metrics_f0.md
               ccc_V = 0.4652
               ccc_A = 0.4067
              CCC_VA = 0.4360
       F1_EXPR_macro = 0.3445
          F1_AU_best = 0.4849
           t_AU_best = 0.5000
               P_MTL = 1.7013
              w_expr = 0.6000
                w_va = 0.9000
                w_au = 0.6000
             variant = f0_grid
F0: {'ccc_V': 0.4652, 'ccc_A': 0.4067, 'CCC_VA': 0.436, 'F1_EXPR_macro': 0.3445, 'F1_AU_best': 0.4849, 't_AU_best': 0.5, 'P_MTL': 1.7013, 'w_expr': 0.6, 'w_va': 0.9, 'w_au': 0.6, 'variant': 'f0_grid'}


In [6]:
VARIANT_PATHS = {
    'visual_only': ('configs/stage3_visual_only.yaml', 'results/stage3_visual_only/best.pt'),
    'audio_only':  ('configs/stage3_audio_only.yaml',  'results/stage3_audio_only/best.pt'),
    'f1_concat':   ('configs/stage3_f1_concat.yaml',   'results/stage3_f1_concat/best.pt'),
    'f2_blend':    ('configs/stage3_f2_blend.yaml',    'results/stage3_f2_blend/best.pt'),
    'f3_gate':     ('configs/stage3_f3_gate.yaml',     'results/stage3_f3_gate/best.pt'),
    'f4_xattn':    ('configs/stage3_f4_xattn.yaml',    'results/stage3_f4_xattn/best.pt'),
    'f5_lmf':      ('configs/stage3_f5_lmf.yaml',      'results/stage3_f5_lmf/best.pt'),
}

all_metrics = {'f0_grid': f0_metrics}
for name, (cfg, ckpt) in VARIANT_PATHS.items():
    m = evaluate_variant(checkpoint=ckpt, config_path=cfg)
    all_metrics[name] = m
    print(f'{name:14s} P_MTL={m.get("P_MTL", float("nan")):.4f}  '
          f'CCC_VA={m.get("CCC_VA", float("nan")):.4f}  '
          f'F1_EXPR={m.get("F1_EXPR_macro", float("nan")):.4f}  '
          f'F1_AU={m.get("F1_AU@0.5", float("nan")):.4f}')

[stage3] wrote results\stage3_visual_only\metrics.md
               ccc_V = 0.4630
               ccc_A = 0.3995
              CCC_VA = 0.4313
       F1_EXPR_macro = 0.3335
            ACC_EXPR = 0.3483
           F1_AU@0.5 = 0.4792
          F1_AU_best = 0.4827
           t_AU_best = 0.6000
           P_MTL@0.5 = 1.2439
          P_MTL_best = 1.2474
             variant = visual_only
    trainable_params = 179706.0000
visual_only    P_MTL=nan  CCC_VA=0.4313  F1_EXPR=0.3335  F1_AU=0.4792
[stage3] wrote results\stage3_audio_only\metrics.md
               ccc_V = 0.2142
               ccc_A = 0.2130
              CCC_VA = 0.2136
       F1_EXPR_macro = 0.1693
            ACC_EXPR = 0.1896
           F1_AU@0.5 = 0.2987
          F1_AU_best = 0.3609
           t_AU_best = 0.3000
           P_MTL@0.5 = 0.6815
          P_MTL_best = 0.7437
             variant = audio_only
    trainable_params = 142998.0000
audio_only     P_MTL=nan  CCC_VA=0.2136  F1_EXPR=0.1693  F1_AU=0.2987
[stage3] wrote r

## 4. Comparison table — Stage 3 vs. Sprint A.5 frame-average baseline

Reuses the JSON dumps under `results/aw2_stage1_{enet,mbf}/baseline_frameavg.json` as the zero-train floor. Comparison is on the AU-blind two-task analogue $P_{\mathrm{MTL}}^{\dagger} = \mathrm{CCC}_{VA} + F_1^{\mathrm{EXPR}}$ since the EmotiEffLib head produces no AU output. The per-task metrics also include AU rows from Stage 3 for completeness.

In [7]:
def _two_task(m):
    """Restricted P_MTL (no AU): CCC_VA + F1_EXPR_macro."""
    ccc_va  = m.get('CCC_VA', float('nan'))
    f1_expr = m.get('F1_EXPR_macro', float('nan'))
    return ccc_va, f1_expr, ccc_va + f1_expr

rows = []
# Frame-average baseline rows (Sprint A.5)
for backbone, tag in [('enet', 'enet_b0_8_va_mtl'), ('mbf', 'mbf_va_mtl')]:
    p = REPO / 'results' / f'aw2_stage1_{backbone}' / 'baseline_frameavg.json'
    if not p.exists():
        print(f'[warn] missing {p}; rerun eval_frame_average_baseline')
        continue
    b = json.loads(p.read_text(encoding='utf-8'))
    for mode in ('frame', 'video'):
        d = b[mode]
        rows.append((
            f'frame-avg/{mode}/{backbone}',
            d['ccc_V'], d['ccc_A'], d['ccc_VA'], d['f1_expr'], None,
            d['p_mtl_va_expr'],
        ))

# Stage 3 rows (single seed; enet visual + hubert audio per the shared default)
for name, m in all_metrics.items():
    ccc_va, f1_expr, p_dag = _two_task(m)
    rows.append((
        f'stage3/{name}',
        m.get('ccc_V', float('nan')), m.get('ccc_A', float('nan')),
        ccc_va, f1_expr, m.get('F1_AU@0.5'), p_dag,
    ))

# Render markdown
lines = ['# Aff-Wild2 Stage 3 vs. frame-average baseline\n']
lines.append('| Method | CCC_V | CCC_A | CCC_VA | F1_EXPR | F1_AU | P_MTL\u2020 (VA+EXPR) |')
lines.append('| --- | --- | --- | --- | --- | --- | --- |')
def fmt(x):
    return '—' if x is None else f'{x:.4f}'
for r in rows:
    name, v, a, va, fe, fa, p = r
    lines.append(f'| `{name}` | {fmt(v)} | {fmt(a)} | {fmt(va)} | {fmt(fe)} | {fmt(fa)} | {fmt(p)} |')
summary_md = '\n'.join(lines) + '\n'

(AW2_S3 / 'summary.md').write_text(summary_md, encoding='utf-8')
(AW2_S3 / 'summary.json').write_text(json.dumps(all_metrics, indent=2, default=float), encoding='utf-8')
print(summary_md)

# Aff-Wild2 Stage 3 vs. frame-average baseline

| Method | CCC_V | CCC_A | CCC_VA | F1_EXPR | F1_AU | P_MTL† (VA+EXPR) |
| --- | --- | --- | --- | --- | --- | --- |
| `frame-avg/frame/enet` | 0.4252 | 0.2444 | 0.3348 | 0.2152 | — | 0.5500 |
| `frame-avg/video/enet` | 0.3599 | 0.2007 | 0.2803 | 0.1102 | — | 0.3905 |
| `frame-avg/frame/mbf` | 0.4380 | 0.2022 | 0.3201 | 0.2245 | — | 0.5446 |
| `frame-avg/video/mbf` | 0.3519 | 0.1643 | 0.2581 | 0.0957 | — | 0.3538 |
| `stage3/f0_grid` | 0.4652 | 0.4067 | 0.4360 | 0.3445 | — | 0.7805 |
| `stage3/visual_only` | 0.4630 | 0.3995 | 0.4313 | 0.3335 | 0.4792 | 0.7647 |
| `stage3/audio_only` | 0.2142 | 0.2130 | 0.2136 | 0.1693 | 0.2987 | 0.3829 |
| `stage3/f1_concat` | 0.4262 | 0.4216 | 0.4239 | 0.3106 | 0.4679 | 0.7345 |
| `stage3/f2_blend` | 0.4477 | 0.3939 | 0.4208 | 0.3394 | 0.4817 | 0.7602 |
| `stage3/f3_gate` | 0.4545 | 0.3952 | 0.4249 | 0.3309 | 0.4832 | 0.7558 |
| `stage3/f4_xattn` | 0.4449 | 0.5125 | 0.4787 | 0.3371 | 0.4831 | 0.8158 |
| 

Drop the contents of `results/aw2_stage3/summary.md` into the §6.x bimodal table in `internship_report.tex`. The expected pattern (echoing the interim CREMA-D leaderboard) is `audio_only < visual_only < F0 \u2264 F2,F5 < F1,F3 < F4`, with the frame-average baseline rows sitting strictly below `visual_only`. If F4 (or whichever variant wins) does not clear the strongest baseline by $\geq 0.20\,P_{\mathrm{MTL}}^{\dagger}$, reframe the bimodal contribution per the v3-plan §3 RQ5 fallback.